# MergerRetriever and LongContextReorder on Google Colab with GPU

## What This Notebook Covers

This notebook ports the MergerRetriever and LongContextReorder multi-source RAG pipeline to Google Colab using a GPU runtime and a quantized GGUF language model. The core retrieval architecture is identical to the local VS Code version, but three things change in this Colab version.

First, Google Drive is mounted so that large persistent files like the GGUF model survive across Colab session resets without re-uploading.

Second, the language model is Zephyr-7B-Beta compressed into a 4-bit GGUF file and run through llama-cpp-python instead of the smaller TinyLlama running through HuggingFace Transformers. This produces significantly higher quality answers at the cost of a larger model file.

Third, the chain is RetrievalQA with return_source_documents=True which makes source attribution visible and lets us verify that answers are drawn from multiple pages across both books.

The notebook is organised into seven categories.

Category 1 installs all required libraries.

Category 2 mounts Google Drive and loads both PDF knowledge bases from the Colab session filesystem.

Category 3 splits all pages into chunks ready for embedding.

Category 4 loads embedding models, retrieves the OpenAI key from Colab secrets, and ingests both books into separate ChromaDB collections.

Category 5 creates individual MMR retrievers for each book and merges them using MergerRetriever (LOTR).

Category 6 builds the compression pipeline with EmbeddingsRedundantFilter and LongContextReorder.

Category 7 loads Zephyr-7B-Beta GGUF on GPU, builds a RetrievalQA chain, and runs queries with full source attribution.

## Why Colab and GGUF Instead of Local TinyLlama

TinyLlama has 1.1 billion parameters and runs comfortably on a laptop CPU. Zephyr-7B-Beta has 7 billion parameters which would be extremely slow on CPU but runs well on a Colab T4 GPU. The 4-bit GGUF quantization compresses the model from roughly 28 gigabytes to roughly 4 gigabytes in memory, making it fit on the GPU. This gives substantially better answer quality while keeping GPU memory requirements manageable. The entire retrieval and compression pipeline stays identical between both versions.

## Category 1  Installing Required Libraries

### Why Each Library Is Needed

langchain is the core framework that provides document loaders, text splitters, retrievers, vector store wrappers, chains, and the retriever interface that ties everything together.

chromadb is a lightweight local vector database that stores document chunk embeddings on disk and runs fast approximate nearest-neighbor search at query time without needing a separate server process.

huggingface_hub and sentence-transformers together provide access to pre-trained embedding models that run entirely inside the Colab runtime without any external API calls or per-token costs.

pypdf is the underlying library that PyPDFLoader uses to parse PDF binary format and extract raw text from each page.

openai is imported because OpenAIEmbeddings is also instantiated in this notebook as an alternative embedding option alongside the HuggingFace models.

tiktoken is a tokenizer used internally by some LangChain prompt templates and chain types when they need to estimate the token count of assembled prompts.

langchain-community provides the third-party integrations that are maintained separately from the core LangChain package, including ChromaDB vector stores, HuggingFace embedding wrappers, and LlamaCpp.

Colab runtimes start completely fresh after every session disconnect, so this entire installation cell must run at the beginning of every new session before any other cell.

In [ ]:
# Install all required libraries for this Colab session
# Colab runtimes reset after disconnection so this cell must run at the start of every session
!pip install -qU langchain chromadb huggingface_hub sentence-transformers pypdf openai tiktoken
!pip install -U langchain-community

## Category 2  Google Drive Setup and Data Loading

### Why Google Drive Must Be Mounted

Colab runtimes are temporary. Every time the runtime disconnects or resets, every file stored only in the Colab session filesystem at /content is permanently lost. The Zephyr-7B-Beta GGUF model file is several gigabytes and cannot be re-uploaded or re-downloaded at the start of every session.

Mounting Google Drive at /content/drive makes the entire Drive filesystem available inside the Colab runtime. The GGUF model stored in Drive can then be loaded from /content/drive/MyDrive/ in every session without any re-upload or re-download. drive.mount will prompt for Google account authorization the first time it runs.

### Why We Need PyPDFLoader

PDF files are stored in binary format, not plain text. PyPDFLoader uses the pypdf library internally to parse that binary format, extract the text content from every page, and return a structured Python list of LangChain Document objects. Each Document carries two attributes: page_content which is the raw extracted text string, and metadata which is a dictionary automatically filled with the source file path and the page number. This metadata is critical because it persists through the entire pipeline and is what lets us trace every retrieved chunk back to its exact page and source file at the end.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# Mount Google Drive so files stored there are accessible under /content/drive/MyDrive/
# This is required for loading the large GGUF model file which cannot be stored in the
# temporary Colab session filesystem since it would be lost after every session reset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Load the Harry Potter PDF from the temporary Colab session filesystem
# This file must be uploaded to /content/ using the Colab file upload panel at the start of each session
# Unlike the GGUF model, small PDF files can be uploaded fresh each session without significant time cost
loader_harrypotter = PyPDFLoader("/content/harry_potter_book.pdf")

In [ ]:
# load() reads every page of the PDF and returns one Document object per page
# Each Document has page_content (the extracted text) and metadata (source path and page number)
documnet_harrypotter = loader_harrypotter.load()

In [ ]:
# Check how many pages were successfully loaded from the Harry Potter PDF
print(len(documnet_harrypotter))

In [ ]:
# Load the Game of Thrones PDF from the temporary Colab session filesystem
loader_got = PyPDFLoader("/content/got_book.pdf")

In [ ]:
documnet_got = loader_got.load()

In [ ]:
# Check how many pages were loaded from the GOT PDF
print(len(documnet_got))

## Category 3  Text Splitting Into Chunks

### Why Splitting Is Necessary

LLMs and embedding models both have fixed maximum token limits. A single PDF page can contain 2000 to 4000 characters which is too long for most embedding models to encode accurately and too coarse a unit for precise retrieval. If an entire page is stored as one chunk, a retrieval query can only return whole pages and may drag in large amounts of irrelevant content alongside the relevant part.

RecursiveCharacterTextSplitter solves this by breaking every page into smaller overlapping pieces. It attempts to split on paragraph boundaries first, then sentence boundaries, then word boundaries, working recursively until each piece is within the configured size. This recursive approach preserves natural language structure better than a naive fixed-window character split.

chunk_size=500 means each chunk contains at most 500 characters of text.

chunk_overlap=100 means the last 100 characters of each chunk are repeated at the start of the next chunk. This overlap ensures that sentences or key phrases that fall exactly at a chunk boundary are still fully present in at least one chunk and cannot be silently cut in half and lost.

# Create the Text Splitter for Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter splits text by trying different separators in order
# chunk_size=500: each chunk contains at most 500 characters
# chunk_overlap=100: the last 100 characters of one chunk are repeated at the start of the next
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
# Split all Harry Potter pages into smaller chunks
# The output is still a list of Document objects, each now with shorter page_content
# Every child chunk inherits the source path and page number from its parent page
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)

In [ ]:
# Split all GOT pages into smaller chunks
text_got = text_splitter.split_documents(documnet_got)

In [ ]:
# Check how many chunks were produced from Harry Potter
# More chunks means finer-grained retrieval at the cost of more embedding calls during ingestion
print(len(text_harrypotter))

In [ ]:
# Check how many chunks were produced from GOT
print(len(text_got))

## Category 4  Embedding Models and ChromaDB Vector Stores

### Three Embedding Models With Different Roles

This notebook imports three different embedding model classes, each chosen for a specific purpose in the pipeline.

HuggingFaceEmbeddings with all-MiniLM-L6-v2 is a fast and lightweight model producing 384-dimensional vectors. It is used later inside the EmbeddingsRedundantFilter in Category 6 because that filter needs to compare many pairs of retrieved chunks quickly. Accuracy matters less than speed in that context since the filter only needs to detect obvious near-duplicates, not perform fine-grained relevance ranking.

HuggingFaceBgeEmbeddings with BAAI/bge-large-en is a more powerful model producing 1024-dimensional vectors. BGE stands for BAAI General Embedding, developed by the Beijing Academy of Artificial Intelligence and specifically optimised for retrieval tasks. It is used for ingesting documents into ChromaDB and building the main retrievers because retrieval quality has the single largest impact on final answer accuracy.

OpenAIEmbeddings is available as a third option that calls OpenAI's hosted embedding API. It is instantiated here but not used as the primary embedding for this pipeline.

### Loading the OpenAI Key From Colab Secrets

Colab provides a built-in secrets manager accessible via the key icon in the left sidebar. Storing the OpenAI API key there instead of hardcoding it in a notebook cell ensures the key never appears in the notebook file itself. userdata.get retrieves a named secret at runtime and makes it available as a Python string. The key is then written into os.environ so that any library reading OPENAI_API_KEY from the environment can authenticate automatically.

### Creating Separate ChromaDB Collections for Each Book

Chroma.from_documents performs three operations internally for every call. It passes every chunk's page_content through the bge embedding model to produce a dense vector. It stores each chunk's text and vector together in the named collection. It builds a searchable HNSW index over all vectors so future nearest-neighbor queries run in milliseconds rather than scanning every stored vector linearly.

Keeping Harry Potter and GOT in two completely separate named collections is essential for MergerRetriever in Category 5. Each collection will be backed by its own retriever, and the merger will query both independently. If both books were mixed into one collection we would lose the ability to tune or inspect each source separately.

collection_metadata with hnsw set to cosine sets cosine similarity as the vector distance metric. Cosine similarity measures only the angle between two vectors and ignores their magnitude, which is the correct choice for text embeddings because vector length varies with document length and should not affect the relevance score.

# Load the Embedding Model to Convert the Data into Vectors

In [ ]:
from langchain_community.embeddings import (
    HuggingFaceEmbeddings,
    OpenAIEmbeddings,
    HuggingFaceBgeEmbeddings,
)

In [ ]:
# Initialize the lightweight MiniLM embedding model
# Used later inside EmbeddingsRedundantFilter for fast pairwise similarity comparison between chunks
hf_minilm_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# Initialize the BGE large embedding model
# Used for document ingestion into ChromaDB and building the main retrievers
# Produces higher quality 1024-dimensional embeddings optimised for retrieval tasks
hf_bge_embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-large-en"
)

In [ ]:
# Retrieve the OpenAI API key from Colab's secure secrets manager
# Add this secret first by clicking the key icon in the Colab left sidebar
# This approach keeps the key out of the notebook file so it is safe to share or push to GitHub
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [ ]:
# Write the key into the OS environment so all libraries that read OPENAI_API_KEY
# from the environment can authenticate automatically without being passed the key explicitly
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
# Instantiate OpenAIEmbeddings as an available alternative embedding option
# Not used as the primary embedding in this notebook but available if needed
openai_embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Now Ingest the Data into the Chroma Database

In [ ]:
from langchain_community.vectorstores import Chroma
import chromadb

In [ ]:
# Check the current working directory inside the Colab runtime
import os
os.getcwd()

In [ ]:
# Resolve the absolute path of the current Colab working directory
# os.path.abspath resolves any relative path components to produce a canonical absolute path
CURRENT_DIR = os.path.dirname(os.path.abspath("."))

In [ ]:
CURRENT_DIR

In [ ]:
# Build the path where ChromaDB will persist its index files during this Colab session
# This lives in the temporary Colab filesystem at /content/db rather than Google Drive
# The vector index will need to be rebuilt each session, but ingestion is fast compared to model loading
DB_DIR = os.path.join(CURRENT_DIR, "/content/db")

In [ ]:
DB_DIR

In [ ]:
# Configure ChromaDB for persistent storage within this Colab session
# is_persistent=True: writes the index to disk so it survives within the same session
# persist_directory: folder where ChromaDB writes all index files
# anonymized_telemetry=False: disables sending usage statistics to Chroma cloud servers
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [ ]:
# Ingest Harry Potter chunks into ChromaDB
# hf_bge_embeddings generates a 1024-dimensional vector for every chunk
# collection_name='harrypotter' namespaces this data separately from the GOT collection
# collection_metadata with hnsw cosine sets cosine similarity as the nearest-neighbor distance metric
harrypotter_vectorstore = Chroma.from_documents(
    text_harrypotter,
    hf_bge_embeddings,
    client_settings=client_settings,
    collection_name="harrypotter",
    collection_metadata={"hnsw": "cosine"},
    persist_directory="/store/harrypotter"
)

In [ ]:
# Ingest GOT chunks into a completely separate ChromaDB collection
# Separate collection_name and persist_directory keep GOT data fully independent from Harry Potter
got_vectorstore = Chroma.from_documents(
    text_got,
    hf_bge_embeddings,
    client_settings=client_settings,
    collection_name="got",
    collection_metadata={"hnsw": "cosine"},
    persist_directory="/store/got"
)

## Category 5  Individual Retrievers and MergerRetriever (LOTR)

### Wrapping Each Vector Store as a Retriever

as_retriever converts a ChromaDB vector store into a LangChain retriever object. This abstraction means the underlying vector store can be swapped for any other retriever type, BM25, Weaviate, Pinecone, or anything else, without changing any code downstream in the pipeline.

search_type='mmr' enables Maximum Marginal Relevance search. Standard similarity search simply returns the top k chunks closest to the query vector, which often produces near-duplicate results because adjacent sections of a book share very similar language. MMR prevents this by iteratively selecting chunks that are both relevant to the query and meaningfully different from chunks already selected. Each newly selected chunk must maximise its contribution of new information rather than repeating what has already been chosen. This produces a more diverse and informative result set from a single book.

k=5 means each retriever returns 5 chunks per query. With two retrievers the raw merged output contains up to 10 chunks before the compression pipeline filters and reorders them.

### Merging Both Retrievers With MergerRetriever

MergerRetriever is also called LOTR, Lord of the Retrievers. It accepts a list of any number of individual retrievers and when queried it sends the same query to all of them simultaneously. The results from every retriever are concatenated into one flat list of Document objects.

This is the architectural component that makes multi-source RAG possible without manually querying each source and combining results. A single call to lotr.invoke(query) automatically retrieves from both books and returns all results together.

The raw merged output is intentionally unfiltered at this stage. Questions about Harry Potter still return GOT chunks and vice versa because the merger does not filter by relevance, only by proximity to the query within each individual collection. Category 6 adds the compression pipeline that addresses this noise.

# Now Create a Retriever

In [ ]:
# Create a retriever for the Harry Potter vector store
# search_type='mmr': Maximum Marginal Relevance produces diverse and relevant results
# k=5: return the top 5 most relevant and diverse chunks per query
retriever_harrypotter = harrypotter_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "include_metadata": True}
)

In [ ]:
# Create a separate retriever for the GOT vector store using identical settings
retriever_got = got_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "include_metadata": True}
)

# Let's Merge both Retriever

# It is also called lord of retriever (LOTR)

In [ ]:
# Print installed library versions before importing MergerRetriever
# MergerRetriever has moved between langchain packages across versions
# Confirming the versions helps diagnose import errors if the package layout has changed
import langchain
import langchain_community

print(langchain.__version__)
print(langchain_community.__version__)

In [ ]:
# Install langchain-classic which hosts MergerRetriever in the package structure
# used by this notebook
pip install -U langchain-classic

In [ ]:
from langchain_classic.retrievers import MergerRetriever

In [ ]:
# Combine both book retrievers into one unified retriever
# When queried, lotr calls retriever_harrypotter and retriever_got simultaneously
# and returns all results concatenated into a single flat list
lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_got])

In [ ]:
# Test the merged retriever with a GOT question
# Notice results from BOTH books appear in the raw output even for a book-specific question
# This is expected at this stage and will be resolved by the compression pipeline in Category 6
for doc in lotr.invoke("Who was Jon Snow?"):
    print(doc.page_content)

In [ ]:
# Test with a Harry Potter question to see the same cross-book mixing in the raw output
for doc in lotr.invoke("Who is a harry potter?"):
    print(doc.page_content)

## See This Result Is Too Messy

The output above mixes chunks from both books regardless of which book the question actually concerns. Some chunks may also be near-duplicates of each other since MMR within one retriever does not know what the other retriever already selected. This noise wastes LLM context window space and can bury the most relevant content in irrelevant surrounding material. The next category builds the compression pipeline that removes redundant chunks and reorders the survivors to overcome the lost-in-the-middle attention problem.

## Category 6  Compression Pipeline With EmbeddingsRedundantFilter and LongContextReorder

### Two Problems Requiring Two Solutions

The raw merged output from LOTR in Category 5 has two structural problems that this pipeline addresses.

Problem 1 is redundancy. When two retrievers independently search overlapping conceptual space some returned chunks will be near-duplicates of each other. Feeding duplicate context to an LLM wastes tokens, can cause the model to overweight that content in its answer, and leaves less room in the context window for genuinely distinct and relevant information.

EmbeddingsRedundantFilter solves this by computing cosine similarity between every pair of retrieved chunks using the hf_bge_embeddings model. Any chunk that is too semantically similar to another already kept chunk is discarded. The result is a deduplicated set where each surviving chunk contributes genuinely new information to the LLM context.

Problem 2 is the lost-in-the-middle phenomenon. Research on LLM attention patterns has consistently shown that models pay the most attention to content at the very beginning and the very end of their context window. Content placed in the middle of a long context is systematically underweighted, even when it is more relevant than the content at the edges.

LongContextReorder solves this by reordering the deduplicated chunks so the highest-relevance chunks appear at position 1 and the last position in the list, while the lowest-relevance chunks are placed in the middle. This positional arrangement ensures the LLM allocates maximum attention to the highest-quality content in the context.

DocumentCompressorPipeline chains both transformers sequentially: EmbeddingsRedundantFilter runs first to remove duplicates, then LongContextReorder runs on the survivors to optimise their positions.

ContextualCompressionRetriever wraps the entire system so the rest of the notebook just calls compression_retriever_reordered.invoke(query) and receives a clean, deduplicated, optimally ordered list of chunks ready for the LLM.

# Now After Understanding Step by Step Create a Pipeline for the LLM

In [ ]:
# Reinstall langchain-classic to ensure the document_transformers submodule is fully available
pip install -U langchain-classic

In [ ]:
from langchain_classic.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
    LongContextReorder,
)

from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline,
)

from langchain_classic.retrievers import ContextualCompressionRetriever

In [ ]:
from re import search

# Step 1: EmbeddingsRedundantFilter removes semantically duplicate chunks
# Uses hf_bge_embeddings to compare every pair of retrieved chunks
# Chunks that are too similar to one already selected are discarded
filter = EmbeddingsRedundantFilter(embeddings=hf_bge_embeddings)

# Step 2: LongContextReorder repositions surviving chunks
# Most relevant chunks move to the start and end of the list
# Least relevant chunks are placed in the middle
# This directly counteracts the lost-in-the-middle attention problem in LLMs
reordering = LongContextReorder()

# Chain both transformers sequentially inside DocumentCompressorPipeline
# filter runs first to remove duplicates, then reordering optimises positions of survivors
pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])

# Wrap the full pipeline behind a single ContextualCompressionRetriever
# base_retriever=lotr: MergerRetriever fetches the initial combined candidates from both books
# base_compressor=pipeline: deduplicates then reorders those candidates
# search_kwargs k=3: return at most 3 chunks after compression for a concise LLM context
compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline,
    base_retriever=lotr,
    search_kwargs={"k": 3, "include_metadata": True}
)

In [ ]:
# Test the compression retriever with a GOT question
# Compare the chunk count and content quality against the raw noisy lotr output from Category 5
# Fewer chunks and higher relevance per chunk should be visible here
docs = compression_retriever_reordered.invoke("Who is Ser Waymar ?")

print(len(docs))
print(docs[0].page_content)

## Category 7  Loading Zephyr-7B-Beta GGUF on GPU and Running the RAG Chain

### Why GGUF and llama-cpp-python on Colab GPU

A full-precision 7 billion parameter model like Zephyr-7B-Beta requires approximately 28 gigabytes of memory in float32, far exceeding the memory available on a standard Colab GPU. The GGUF file format stores model weights in quantized form, here using Q4_0 quantization which represents every weight using only 4 bits instead of the standard 32. This reduces memory requirements to roughly 4 to 5 gigabytes while retaining most of the model's answer quality, making it fit comfortably on a Colab T4 GPU.

llama-cpp-python provides Python bindings for llama.cpp, a highly optimized C++ inference engine designed specifically for running GGUF models efficiently. When compiled with CUDA support on a Colab GPU runtime, llama.cpp can offload compute-intensive matrix multiplications to the GPU, achieving inference speeds that would be impractically slow on CPU alone.

### Why the Model Loads From Google Drive

The GGUF model file is several gigabytes. Downloading or uploading it fresh at the start of every Colab session would take significant time. Because Google Drive was mounted in Category 2, the model can be loaded directly from /content/drive/MyDrive/zephyr-7b-beta.Q4_0.gguf in every session without any wait beyond Drive access latency.

### LlamaCpp Parameter Explanations

streaming=True and stream=True together enable token-by-token streaming output. The model sends each generated token back as soon as it is produced rather than waiting for the entire response to be complete before returning anything. This makes the response feel faster and more interactive.

max_tokens=1500 caps the total length of the generated response to 1500 tokens, preventing runaway generation on open-ended questions.

temperature=0.75 controls sampling randomness. Lower values like 0.2 produce more focused and deterministic output. Higher values like 1.0 produce more varied and creative output. 0.75 is a balanced value that produces natural-sounding answers without too much unpredictability.

top_p=1 means nucleus sampling is fully disabled. The model samples from the complete probability distribution at every generation step rather than restricting sampling to only the top-p fraction of the distribution.

gpu_layers=0 uses llama.cpp's own internal layer offloading count rather than GPU-based offloading via the higher-level parameter. Increasing this value would offload more transformer layers to GPU memory and accelerate generation on a Colab GPU runtime.

n_threads set to half the available CPU cores balances generation speed against leaving resources available for the rest of the Colab environment.

n_ctx=4096 sets the model context window in tokens. This determines how much total text, retrieved chunks plus the question plus any system prompt, the model can process in one call.

### The RetrievalQA Chain

RetrievalQA.from_chain_type connects the compression retriever to the LLM in a single object. When invoked with a question it automatically sends the question to compression_retriever_reordered, receives deduplicated and reordered chunks from both books, assembles a prompt with all chunks as context, calls the LLM to generate an answer, and returns both the answer and the source chunks.

chain_type='stuff' means all retrieved chunks are concatenated directly into one prompt. This is the simplest approach and works well when the combined chunk text fits within n_ctx=4096 tokens.

return_source_documents=True means every response dictionary also contains the exact Document objects used to generate the answer. This is essential for verifying that the answer is grounded in real source content rather than in the model's training data.

# Load the Model from HuggingFace

In [ ]:
# Install llama-cpp-python for running quantized GGUF models inside the Colab runtime
# On a Colab GPU runtime this installs with CUDA support enabling GPU-accelerated inference
!pip install llama-cpp-python

In [ ]:
# Verify the GGUF model file exists at the expected Google Drive path before attempting to load it
# This requires Google Drive to already be mounted from Category 2
# Prints True if the file is found, False if it is missing or Drive is not mounted
model_path = "/content/drive/MyDrive/zephyr-7b-beta.Q4_0.gguf"

print(os.path.exists(model_path))

In [ ]:
# Load the quantized Zephyr-7B-Beta model through llama-cpp-python
# LlamaCpp wraps llama.cpp as a LangChain-compatible LLM object
from langchain_community.llms import LlamaCpp

llms = LlamaCpp(
    streaming=True,
    model_path="/content/drive/MyDrive/zephyr-7b-beta.Q4_0.gguf",
    max_tokens=1500,         # Maximum tokens to generate per response
    temperature=0.75,        # Sampling temperature: 0.75 balances focus and natural variation
    top_p=1,                 # No nucleus sampling filtering, full distribution used
    gpu_layers=0,            # llama.cpp internal layer offload count for GPU acceleration
    stream=True,             # Stream tokens as generated rather than waiting for full output
    verbose=True,            # Print llama.cpp internal logs for debugging load and generation
    n_threads=int(os.cpu_count() / 2),  # Use half available CPU cores for generation
    n_ctx=4096               # Context window in tokens: input plus output must fit within this
)

In [ ]:
from langchain_classic.chains import RetrievalQA

In [ ]:
# Build the RetrievalQA chain connecting the compression retriever to Zephyr-7B-Beta
# chain_type='stuff': all retrieved chunks are concatenated into one prompt
# retriever=compression_retriever_reordered: the full pipeline LOTR plus filter plus reorder
# return_source_documents=True: include the actual chunks used to generate the answer
qa = RetrievalQA.from_chain_type(
    llm=llms,
    chain_type="stuff",
    retriever=compression_retriever_reordered,
    return_source_documents=True
)

### Running Queries and Inspecting Source Attribution

We now run three queries through the complete pipeline. Each invoke call triggers the full sequence: the compression retriever calls both ChromaDB collections via MergerRetriever, deduplicates and reorders the combined results, assembles the context prompt, sends it to Zephyr-7B-Beta for answer generation, and returns both the answer text and the source Document objects used to generate it. Printing the page metadata from each source document proves the answer was grounded across multiple pages from multiple books rather than hallucinated from model training data.

In [ ]:
# Query 1: GOT specific question
# The answer and the source chunks used to construct it are both returned
query = "who is jon snow?"
results = qa(query)
print(results['result'])

print(results["source_documents"])

Jon Snow is a fictional character from the A Song of Ice and Fire series of fantasy novels by George R. R. Martin, as well as its television adaptation, Game of Thrones. He appears as a major point-of-view character in all six novels published so far.

Harry Potter is the main character in J.K. Rowling's series of novels and films about magic and wizards. He is an orphan who discovers that he has magical powers and goes to attend Hogwarts School of Witchcraft and Wizardry, where he makes friends and confronts dark forces throughout the series.

In [ ]:
# Query 2: Harry Potter specific question
# Source metadata is printed at the end to show which pages contributed to this answer
results = qa("who is a harry potter?")
print(results['result'])

print(results["source_documents"])

for source in results["source_documents"]:
    print(source.metadata)

In [ ]:
# Query 3: Complex cross-document question testing how well the pipeline handles
# a nuanced question requiring information spread across multiple chunks and pages
results = qa.invoke(
    "How does Jon Snow's relationship with the Stark family influence his identity and "
    "decisions throughout A Game of Thrones?"
)

In [ ]:
# Inspect the full result dictionary structure
# Contains 'query', 'result', and 'source_documents' keys
results

In [ ]:
# Print just the generated answer text
print(results['result'])

In [ ]:
# Print the raw source Document objects that provided the context for this answer
print(results["source_documents"])

In [ ]:
# Print the full metadata dict for each source document
# Shows source file path, page number, and any other metadata the loader attached
for source in results["source_documents"]:
    print(source.metadata)

In [ ]:
# Print only the page label for each source document
# This gives a clean readable list of every page across both books that contributed to the answer
# Seeing page numbers from both books confirms MergerRetriever is drawing from multiple sources
for source in results["source_documents"]:
    print(f"Page {source.metadata['page_label']}")

## Conclusion

In this implementation a RetrievalQA pipeline was built using MergerRetriever combined with LongContextReorder, running on Google Colab with GPU acceleration and a quantized Zephyr-7B-Beta GGUF model loaded from Google Drive.

When the query "Who is Jon Snow?" was asked, the retriever searched across multiple documents, merged the retrieved chunks from both the Harry Potter and GOT ChromaDB collections, passed them through EmbeddingsRedundantFilter to remove semantic duplicates, reordered them using LongContextReorder to reduce the lost-in-the-middle problem, and passed the optimised context to Zephyr-7B-Beta which generated the final answer.

The LLM generated its answer using the reordered context, while return_source_documents=True returned the exact supporting document chunks used. By printing the page labels from those source documents the retrieved pages were confirmed as Page 2, Page 8, Page 1, Page 2, Page 3, Page 2, Page 1, Page 5, Page 4, and Page 5.

This verified that the answer was generated using information retrieved from multiple pages across different PDF documents rather than relying on a single source. This demonstrates that MergerRetriever improves retrieval coverage by combining results from multiple independent retrievers, and LongContextReorder improves context organisation before passing it to the LLM, leading to more accurate and reliable RAG responses for long documents and multi-document knowledge bases.

Running this same architecture on Colab GPU with the larger Zephyr-7B-Beta model instead of the local TinyLlama setup also demonstrates that the MergerRetriever and LongContextReorder pipeline design is fully portable across both local CPU environments and cloud GPU environments, and across both small and large language models, while keeping the retrieval and compression logic completely unchanged between both versions.